# Unit 4 code companion: Prediction and Choosing Predictors

Predict a new case, put the right interval on it, measure error honestly, and compare a few sets of predictors. Short cells, meant to be run one at a time.

Run the cells in order. Each section matches a moment in the slides. The point is to see the mechanism move when you change the inputs, so change them.

## Setup



In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split

jobs = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/moving_jobs.csv")
jobs.head()

,hours,volume_cuft,crew_size,miles,stairs_flights,packing_service,weekend,est_boxes,dispatcher_rating,quote_source,truck_id
0,7.83,557,3,7.9,1,0,0,37,5,web,T-5
1,10.88,662,2,23.6,2,0,1,51,5,web,T-4
2,16.00,1066,3,3.9,4,0,1,101,2,phone,T-2
3,11.37,845,2,4.7,0,0,0,76,2,referral,T-5
4,7.37,339,3,5.7,2,1,0,21,3,web,T-3


## 1. Fit the model



In [2]:
fit = smf.ols("hours ~ volume_cuft + crew_size + stairs_flights + miles + packing_service",
              data=jobs).fit()
fit.params.round(4)

Intercept          2.3213
volume_cuft        0.0125
crew_size         -0.5292
stairs_flights     0.3284
miles              0.0113
packing_service    0.8976
dtype: float64

## 2. Predict one new job



A new customer: 800 cubic feet, a crew of 3, two flights of stairs, 12 miles, and packing.

In [3]:
new = pd.DataFrame({"volume_cuft": [800], "crew_size": [3], "stairs_flights": [2],
                    "miles": [12], "packing_service": [1]})
fit.predict(new)

0    12.391504
dtype: float64

## 3. The two intervals



In [4]:
fit.get_prediction(new).summary_frame(alpha=0.05).round(2)

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,12.39,0.07,12.25,12.54,10.74,14.04


`mean_ci_lower` and `mean_ci_upper` are the **confidence interval** for the average job like this one. `obs_ci_lower` and `obs_ci_upper` are the **prediction interval** for this one job. The prediction interval is the wide one, and it is the one a dispatcher needs.

## 4. Is the new job inside the data?

Each value on its own, and then the combination.

In [5]:
print(jobs[["volume_cuft", "crew_size", "miles"]].describe().loc[["min", "max"]])

near = jobs[(jobs.volume_cuft.between(700, 900)) & (jobs.crew_size == 3)]
print(f"\ncompleted jobs with a similar volume AND the same crew size: {len(near)}")

     volume_cuft  crew_size  miles
min        120.0        2.0    1.0
max       1317.0        4.0   47.2

completed jobs with a similar volume AND the same crew size: 66


Both checks matter. A value can be ordinary on its own while the combination never happened, and the model gives no warning when that is the case.

## 5. Error on your own rows is too small



In [6]:
X = jobs[["volume_cuft", "crew_size", "stairs_flights", "miles", "packing_service"]]
y = jobs["hours"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

m = LinearRegression().fit(X_train, y_train)
print("error on the rows it was fitted on:", round(np.sqrt(((y_train - m.predict(X_train))**2).mean()), 2))
print("error on the rows held back      :", round(np.sqrt(((y_test - m.predict(X_test))**2).mean()), 2))

error on the rows it was fitted on: 0.79
error on the rows held back      : 0.93


## 6. Cross-validation, so every row gets held out once



In [7]:
folds = KFold(5, shuffle=True, random_state=0)
mse = -cross_val_score(LinearRegression(), X, y, cv=folds, scoring="neg_mean_squared_error")
print("error in each of the five rounds:", np.sqrt(mse).round(3))
print("average:", np.sqrt(mse.mean()).round(3), "hours")

error in each of the five rounds: [0.929 0.881 0.836 0.727 0.811]
average: 0.84 hours


## 7. Compare a few sets of predictors

One branch office, 60 jobs, where an extra column actually costs something.

In [8]:
branch = jobs.sample(60, random_state=5)

sets = {
    "volume only": ["volume_cuft"],
    "the five that make sense": ["volume_cuft", "crew_size", "stairs_flights",
                                 "miles", "packing_service"],
    "those five plus three junk": ["volume_cuft", "crew_size", "stairs_flights",
                                   "miles", "packing_service",
                                   "est_boxes", "dispatcher_rating", "weekend"],
}
for name, cols in sets.items():
    mse = -cross_val_score(LinearRegression(), branch[cols], branch.hours, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    print(f"{name:<28} cross-validated error {np.sqrt(mse):.3f}")

volume only                  cross-validated error 1.090
the five that make sense     cross-validated error 1.012
those five plus three junk   cross-validated error 1.061


The junk columns make it worse. On all 600 jobs they would cost almost nothing, which is why the size of your data decides how much you can afford.

## 8. The same three sets, by R-squared and AIC



In [9]:
for name, cols in sets.items():
    f = smf.ols("hours ~ " + " + ".join(cols), data=branch).fit()
    print(f"{name:<28} R2 {f.rsquared:.4f}   adj R2 {f.rsquared_adj:.4f}   AIC {f.aic:.1f}")

volume only                  R2 0.8935   adj R2 0.8917   AIC 177.5
the five that make sense     R2 0.9233   adj R2 0.9162   AIC 165.9
those five plus three junk   R2 0.9246   adj R2 0.9128   AIC 170.8


$R^2$ is highest for the model with the junk in it. Adjusted $R^2$ and AIC both prefer the five that make sense. $R^2$ alone cannot choose a model.

## 9. A squared term is just another column

Adding powers of a predictor is polynomial regression, and it is still ordinary least squares underneath.

In [10]:
straight = ["volume_cuft", "crew_size", "stairs_flights", "miles", "packing_service"]
branch2 = branch.assign(volume_sq=branch.volume_cuft**2)

for name, cols in [("five predictors", straight),
                   ("plus volume squared", straight + ["volume_sq"])]:
    mse = -cross_val_score(LinearRegression(), branch2[cols], branch2.hours, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    print(f"{name:<22} cross-validated error {np.sqrt(mse):.3f}")

five predictors        cross-validated error 1.012
plus volume squared    cross-validated error 1.049


The relationship really does bend, but with 60 jobs there is not enough data to pay for the bend. With all 600 the squared term earns its place.

## 10. Lasso, which shrinks and drops



In [11]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

cols = ["volume_cuft", "crew_size", "stairs_flights", "miles", "packing_service",
        "est_boxes", "dispatcher_rating", "weekend"]
Z = StandardScaler().fit_transform(branch[cols])

alphas = np.logspace(-3, 0.7, 80)          # the penalties to try
las = LassoCV(alphas=alphas, cv=5, random_state=0, max_iter=50000).fit(Z, branch.hours)
print("penalty chosen by cross-validation:", round(las.alpha_, 4))
print(pd.Series(las.coef_.round(3), index=cols).to_string())

penalty chosen by cross-validation: 0.0254
volume_cuft          2.853
crew_size           -0.244
stairs_flights       0.190
miles                0.195
packing_service      0.393
est_boxes            0.028
dispatcher_rating   -0.082
weekend             -0.018


At the penalty with the best error the junk columns survive with coefficients near zero. Push the penalty higher and they go to exactly zero before any real predictor does.